In [1]:
# ================================
# PART 1: LSTM TEXT CLASSIFICATION
# ================================

import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("----- LSTM TEXT CLASSIFICATION -----")

# Load dataset
vocab_size = 10000
max_len = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

# Padding
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

# Model
model = Sequential([
    Embedding(vocab_size, 128, input_length=max_len),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
model.fit(
    x_train, y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.2
)

# Evaluate
loss, acc = model.evaluate(x_test, y_test)
print("LSTM Accuracy:", acc)

----- LSTM TEXT CLASSIFICATION -----
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


313/313 ━━━━━━━━━━━━━━━━━━━━ 88s 273ms/step - accuracy: 0.8009 - loss: 0.4231 - val_accuracy: 0.8666 - val_loss: 0.3266
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 141s 270ms/step - accuracy: 0.8984 - loss: 0.2544 - val_accuracy: 0.8728 - val_loss: 0.3241
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 148s 289ms/step - accuracy: 0.9301 - loss: 0.1833 - val_accuracy: 0.8614 - val_loss: 0.4339
782/782 ━━━━━━━━━━━━━━━━━━━━ 27s 35ms/step - accuracy: 0.8580 - loss: 0.4517
LSTM Accuracy: 0.8580399751663208


In [2]:
# ==========================================
# SEQ2SEQ MODEL (FINAL WORKING VERSION)
# ==========================================

import tensorflow as tf
import numpy as np
import os
import zipfile

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

print("TensorFlow Version:", tf.__version__)

# ==========================================
# STEP 1: DOWNLOAD + EXTRACT DATASET
# ==========================================

print("\nDownloading dataset...")

zip_path = tf.keras.utils.get_file(
    fname="fra-eng.zip",
    origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip"
)

extract_dir = os.path.dirname(zip_path)

# Force extract every time (avoids missing file issue)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Find fra.txt automatically
data_path = None
for root, dirs, files in os.walk(extract_dir):
    if "fra.txt" in files:
        data_path = os.path.join(root, "fra.txt")
        break

if data_path is None:
    raise FileNotFoundError("fra.txt not found!")

print("Dataset found at:", data_path)

# ==========================================
# STEP 2: LOAD DATA
# ==========================================

lines = open(data_path, encoding="utf-8").read().split("\n")

input_texts = []
target_texts = []

for line in lines[:5000]:   # reduce if slow
    parts = line.split("\t")

    if len(parts) < 2:
        continue

    eng = parts[0]
    fra = parts[1]

    input_texts.append(eng)
    target_texts.append("\t " + fra + " \n")

print("Total samples:", len(input_texts))

# ==========================================
# STEP 3: TOKENIZATION
# ==========================================

input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)

input_seq = input_tokenizer.texts_to_sequences(input_texts)
max_encoder_len = max(len(seq) for seq in input_seq)
input_seq = pad_sequences(input_seq, maxlen=max_encoder_len, padding='post')

target_tokenizer = Tokenizer(filters='')
target_tokenizer.fit_on_texts(target_texts)

target_seq = target_tokenizer.texts_to_sequences(target_texts)
max_decoder_len = max(len(seq) for seq in target_seq)
target_seq = pad_sequences(target_seq, maxlen=max_decoder_len, padding='post')

num_encoder_tokens = len(input_tokenizer.word_index) + 1
num_decoder_tokens = len(target_tokenizer.word_index) + 1

# ==========================================
# STEP 4: PREPARE DATA
# ==========================================

decoder_input_data = target_seq[:, :-1]
decoder_target_data = target_seq[:, 1:]

decoder_target_data = tf.keras.utils.to_categorical(
    decoder_target_data,
    num_classes=num_decoder_tokens
)

# ==========================================
# STEP 5: BUILD MODEL
# ==========================================

latent_dim = 256

# Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(num_encoder_tokens, latent_dim)(encoder_inputs)
_, state_h, state_c = LSTM(latent_dim, return_state=True)(enc_emb)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(num_decoder_tokens, latent_dim)(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Final model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ==========================================
# STEP 6: TRAIN MODEL
# ==========================================

model.fit(
    [input_seq, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=5,
    validation_split=0.2
)

print("\nSeq2Seq Training Completed Successfully!")

TensorFlow Version: 2.20.0

3423204/3423204 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Dataset found at: /root/.keras/datasets/fra.txt
Total samples: 5000


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │    323,072 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 256) │    869,120 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 256),     │    525,312 │ embedding_1[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, None,     │    525,312 │ embedding_2[0][0… │
│                     │ 256), (None,      │            │ lstm_1[0][1],     │
│                     │ 256), (None,      │            │ lstm_1[0][2]      │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None,      │    872,515 │ lstm_2[0][0]      │
│                     │ 3395)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,115,331 (11.88 MB)

 Trainable params: 3,115,331 (11.88 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 76s 1s/step - accuracy: 0.6262 - loss: 3.2380 - val_accuracy: 0.6353 - val_loss: 2.4097
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.7314 - loss: 1.8798 - val_accuracy: 0.6881 - val_loss: 2.2256
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.7450 - loss: 1.7105 - val_accuracy: 0.6890 - val_loss: 2.1355
Epoch 4/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.7470 - loss: 1.5949 - val_accuracy: 0.6964 - val_loss: 2.0516
Epoch 5/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.7558 - loss: 1.4941 - val_accuracy: 0.7013 - val_loss: 2.0042

Seq2Seq Training Completed Successfully!
